# 03 — H→WW EFT Fits (1D and 2D)

Performs EFT scans using reweighting response from coffea and H→WW single-observable likelihood.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
import importlib
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

manifest=json.loads((NOTEBOOK_DIR/'asset_manifest.json').read_text())
coffea_path=manifest.get('default_coffea')
if not coffea_path:
    raise RuntimeError('No default coffea found; run notebook 01 and/or edit manifest')

P_ANALYSIS=Path('/uscms_data/d3/azhou/smeft/analysis')
HBB=P_ANALYSIS/'hbb-coffea'
for x in [P_ANALYSIS,HBB,NOTEBOOK_DIR]:
    if str(x) not in sys.path: sys.path.append(str(x))
import stxs_functions as sf
importlib.reload(sf)
print('using coffea:',coffea_path)

In [ ]:
meas=json.loads((NOTEBOOK_DIR/'hww_measurement.json').read_text())
MU_OBS=meas['mu_obs']; SIGMA_UP=meas['sigma_up']; SIGMA_DN=meas['sigma_dn']

def chi2_hww(mu_pred):
    d=mu_pred-MU_OBS
    s=SIGMA_UP if d>=0 else SIGMA_DN
    return (d/s)**2

def load_sumw_from_fullpath(full):
    from coffea import util
    h=util.load(full)
    return h['sumw_all_noEW'].value

def load_htxs_from_fullpath(full):
    from coffea import util
    return util.load(full)['htxs']

In [ ]:
# Core model helpers
def stxs_coeffs_from_coffea_path(coffea_full_path, operator_list):
    # stxs_functions expects a name in local coffea/; here we reproduce directly
    from coffea import util
    all_h=util.load(coffea_full_path)['htxs']
    wc_mapping=sf.wc_map_dict(operator_list)
    b1,b2={},{}
    for p,wclab in wc_mapping.items():
        if wclab not in list(all_h.axes['wc']):
            continue
        hslice=all_h[{ 'wc': wclab }]
        y1,y2=sf.get_bin_yields(hslice)
        b1[p]=y1; b2[p]=y2
    pts=[p for p in wc_mapping.keys() if p in b1 and p in b2]
    if len(operator_list)==1:
        x=np.array([p[0] for p in pts]); y1=np.array([b1[p] for p in pts]); y2=np.array([b2[p] for p in pts])
        c1,_=sf.curve_fit(sf.quad_1D, x, y1, p0=[1,1,1])
        c2,_=sf.curve_fit(sf.quad_1D, x, y2, p0=[1,1,1])
    elif len(operator_list)==2:
        x1=np.array([p[0] for p in pts]); x2=np.array([p[1] for p in pts]); y1=np.array([b1[p] for p in pts]); y2=np.array([b2[p] for p in pts])
        c1,_=sf.curve_fit(sf.quad_2D, (x1,x2), y1, p0=[1,1,1,1,1,1])
        c2,_=sf.curve_fit(sf.quad_2D, (x1,x2), y2, p0=[1,1,1,1,1,1])
    else:
        raise ValueError('Only 1D/2D supported')
    return c1,c2

In [ ]:
def scan_1d_hww(coffea_full_path, op, sigma_sm_hww, wc_min=-20, wc_max=20, n=401, mg_sigma_pb=3.594):
    sumw=load_sumw_from_fullpath(coffea_full_path)
    c1,c2=stxs_coeffs_from_coffea_path(coffea_full_path,[op])
    rows=[]
    for x in np.linspace(wc_min,wc_max,n):
        sig=(mg_sigma_pb*1000*sf.quad_1D(x,*c1)/sumw) + (mg_sigma_pb*1000*sf.quad_1D(x,*c2)/sumw)
        mu=sig/sigma_sm_hww
        rows.append({'op':op,'wc':float(x),'sigma_pred_fb':float(sig),'mu_pred':float(mu),'chi2_hww':float(chi2_hww(mu))})
    return pd.DataFrame(rows)

def best_fit_from_df(df):
    i=df['chi2_hww'].idxmin()
    return df.loc[i]

In [ ]:
# Example usage (edit op and sigma_sm_hww):
# df=scan_1d_hww(manifest['default_coffea'], op='cHW', sigma_sm_hww=1.0)
# bf=best_fit_from_df(df); print(bf)
# ax=df.plot(x='wc',y='chi2_hww',title='HWW chi2 scan'); ax.axhline(df['chi2_hww'].min()+1,color='r',ls='--')

In [ ]:
def scan_2d_hww(coffea_full_path, ops, sigma_sm_hww, wc_min=-10, wc_max=10, n=121, mg_sigma_pb=3.594):
    if len(ops)!=2: raise ValueError('ops must contain two operators')
    sumw=load_sumw_from_fullpath(coffea_full_path)
    c1,c2=stxs_coeffs_from_coffea_path(coffea_full_path,ops)
    g=np.linspace(wc_min,wc_max,n)
    rows=[]
    for x in g:
        for y in g:
            sig=(mg_sigma_pb*1000*sf.quad_2D((x,y),*c1)/sumw)+(mg_sigma_pb*1000*sf.quad_2D((x,y),*c2)/sumw)
            mu=sig/sigma_sm_hww
            rows.append({'op1':ops[0],'op2':ops[1],'wc1':float(x),'wc2':float(y),'sigma_pred_fb':float(sig),'mu_pred':float(mu),'chi2_hww':float(chi2_hww(mu))})
    return pd.DataFrame(rows)

In [ ]:
# Persist helper outputs
Path('outputs').mkdir(exist_ok=True)
print('ready for scans; outputs dir:', Path('outputs').resolve())